# TASK1 量化交易初体验：从零搭建数据引擎

股票：贵州茅台（600519.SH）  
市场：上海证券交易所 A 股主板

本 Notebook 使用 Tushare 获取过去一年日行情数据，完成清洗、数据质量检查和收盘价曲线绘制。

## 量化交易的优势与限制

优势：能处理更多数据；执行更一致；能够进行历史检验；风险规则可以写进系统。

限制：数据可能出错；历史规律可能失效；模型可能过拟合；真实交易与回测可能不同。

课程核心观点：量化交易不是机器比人聪明，而是把交易想法变成可以重复执行和检查的规则。量化不消灭人的判断，而是把判断从“今天凭感觉买不买”移动到“规则是否合理、证据是否可靠”。

## 基本概念

- K线：一根日 K 线由开盘价、最高价、最低价、收盘价组成，是一个交易日价格路径的压缩快照；成交量和成交额补充反映交易活跃程度。
- 基本面：关注企业经营与估值，例如收入、利润、现金流、行业景气、市盈率、市净率等。
- 技术面：关注价格、成交量以及由它们衍生出的形态或指标，例如均线、涨跌幅、波动率等。
- 基本面与技术面的关系：二者不是互相否定，而是信息来源不同。基本面提供价值判断，技术面提供时机判断，结合使用可以更全面理解市场。

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import tushare as ts

TS_CODE = "600519.SH"
STOCK_NAME = "贵州茅台"
BASE_DIR = Path.cwd()
CSV_PATH = BASE_DIR / "600519_SH_daily_data.csv"
PNG_PATH = BASE_DIR / "close_price_curve.png"
HTML_PATH = BASE_DIR / "600519_SH_dashboard.html"

token = os.getenv("TUSHARE_TOKEN", "YOUR_TUSHARE_TOKEN")
if token == "YOUR_TUSHARE_TOKEN":
    raise RuntimeError("请先设置环境变量 TUSHARE_TOKEN，再运行 Notebook。")
ts.set_token(token)
pro = ts.pro_api()

In [ ]:
start_date = "20250703"
end_date = "20260703"

try:
    df = pro.daily(ts_code=TS_CODE, start_date=start_date, end_date=end_date)
except Exception as exc:
    print(f"Tushare Pro daily 权限不足，使用公开 A 股日行情备用接口。原始错误：{exc}")
    import akshare as ak
    last_error = None
    ak_raw = None
    for attempt in range(3):
        try:
            ak_raw = ak.stock_zh_a_hist(symbol="600519", period="daily", start_date=start_date, end_date=end_date, adjust="")
            break
        except Exception as fallback_exc:
            import time
            last_error = fallback_exc
            time.sleep(2 + attempt)
    if ak_raw is not None and not ak_raw.empty:
        df = pd.DataFrame({
            "trade_date": pd.to_datetime(ak_raw["日期"]).dt.strftime("%Y%m%d"),
            "open": ak_raw["开盘"],
            "high": ak_raw["最高"],
            "low": ak_raw["最低"],
            "close": ak_raw["收盘"],
            "pre_close": ak_raw["收盘"].shift(1),
            "change": ak_raw["涨跌额"],
            "pct_chg": ak_raw["涨跌幅"],
            "vol": ak_raw["成交量"],
            "amount": ak_raw["成交额"],
        })
        df["pre_close"] = df["pre_close"].fillna(df["close"] - df["change"])
    elif CSV_PATH.exists():
        print(f"备用接口临时失败，读取本地 CSV 继续。备用错误：{last_error}")
        df = pd.read_csv(CSV_PATH)
    else:
        raise RuntimeError(f"未获取到日行情数据，备用错误：{last_error}")

if df is None or df.empty:
    raise RuntimeError("未获取到日行情数据。")
df = df[["trade_date", "open", "high", "low", "close", "pre_close", "change", "pct_chg", "vol", "amount"]].copy()
df["trade_date"] = pd.to_datetime(df["trade_date"].astype(str).str.replace("-", "", regex=False), format="%Y%m%d")
df = df.sort_values("trade_date").reset_index(drop=True)
df.to_csv(CSV_PATH, index=False, encoding="utf-8-sig")
df.head(), df.tail(), df.shape

## 数据质量检查原则

1. 可视化是检查工具之一，不是质量证明。
2. 异常是调查线索，不等于垃圾。
3. 智能体可以帮助找问题，但是否可用必须由人负责判断。

In [ ]:
def make_quality_table(df):
    price_cols = ["open", "high", "low", "close"]
    missing = int(df.isna().sum().sum())
    duplicate_dates = int(df["trade_date"].duplicated().sum())
    sorted_ok = bool(df["trade_date"].is_monotonic_increasing)
    non_positive_prices = int((df[price_cols] <= 0).sum().sum())
    negative_volume_amount = int(((df[["vol", "amount"]] < 0).sum()).sum())
    ohlc_bad = int((~(
        (df["high"] >= df["open"]) &
        (df["high"] >= df["close"]) &
        (df["low"] <= df["open"]) &
        (df["low"] <= df["close"]) &
        (df["high"] >= df["low"])
    )).sum())
    yn = lambda x: "通过" if x else "不通过"
    return pd.DataFrame([
        ["数据行数是否合理", "统计记录数并与过去一年交易日经验范围比较", f"{len(df)} 行", yn(len(df) > 150), "行数大于 150，基本合理。"],
        ["是否存在缺失值", "isna().sum()", f"缺失值总数 {missing}", yn(missing == 0), "若出现缺失，应回查来源。"],
        ["是否存在重复交易日期", "trade_date.duplicated()", f"重复日期 {duplicate_dates} 个", yn(duplicate_dates == 0), "重复日期需定位来源。"],
        ["日期是否按升序排列", "is_monotonic_increasing", "已升序" if sorted_ok else "未升序", yn(sorted_ok), "升序便于后续计算。"],
        ["OHLC 是否存在非正数", "open/high/low/close <= 0", f"非正价格 {non_positive_prices} 个", yn(non_positive_prices == 0), "非正价格应核查。"],
        ["成交量和成交额是否存在负数", "vol/amount < 0", f"负数 {negative_volume_amount} 个", yn(negative_volume_amount == 0), "负值应核查。"],
        ["OHLC 逻辑是否合理", "high、low 与 open/close 的逻辑关系", f"异常记录 {ohlc_bad} 行", yn(ohlc_bad == 0), "异常是线索，不应直接删除。"],
    ], columns=["检查项目", "检查方法", "检查结果", "是否通过", "说明 / 处理方式"])

quality_df = make_quality_table(df)
quality_df

In [ ]:
plt.rcParams["font.sans-serif"] = ["Songti SC", "STHeiti", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig, ax = plt.subplots(figsize=(11, 5.8), dpi=160)
ax.plot(df["trade_date"], df["close"], color="#1f6f8b", linewidth=2)
ax.set_title("贵州茅台（600519.SH）过去一年每日收盘价走势")
ax.set_xlabel("交易日期")
ax.set_ylabel("收盘价（元）")
ax.text(0.02, 0.92, "贵州茅台 600519.SH", transform=ax.transAxes)
ax.grid(True, linestyle="--", alpha=0.35)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
fig.autofmt_xdate(rotation=25)
fig.tight_layout()
fig.savefig(PNG_PATH, bbox_inches="tight")
plt.close(fig)
PNG_PATH

In [ ]:
from html import escape

def make_dashboard(df, quality_df):
    actual_start = df["trade_date"].min().strftime("%Y-%m-%d")
    actual_end = df["trade_date"].max().strftime("%Y-%m-%d")
    first_close = float(df["close"].iloc[0])
    last_close = float(df["close"].iloc[-1])
    change_pct = (last_close / first_close - 1) * 100
    max_close = float(df["close"].max())
    min_close = float(df["close"].min())
    avg_close = float(df["close"].mean())
    max_date = df.loc[df["close"].idxmax(), "trade_date"].strftime("%Y-%m-%d")
    min_date = df.loc[df["close"].idxmin(), "trade_date"].strftime("%Y-%m-%d")
    rows = "\n".join(
        f"<tr><td>{escape(str(row['检查项目']))}</td><td>{escape(str(row['是否通过']))}</td>"
        f"<td>{escape(str(row['检查结果']))}；{escape(str(row['说明 / 处理方式']))}</td></tr>"
        for _, row in quality_df.iterrows()
    )
    html = f'''<!doctype html>
<html lang="zh-CN">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>贵州茅台（600519.SH）量化交易数据引擎看板</title>
  <style>
    body {{ margin: 0; background: #f4f7fb; color: #18212f; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", "PingFang SC", "Microsoft YaHei", Arial, sans-serif; line-height: 1.6; }}
    main {{ width: min(1120px, calc(100% - 32px)); margin: 0 auto; padding: 28px 0 36px; }}
    header {{ padding: 24px; color: #fff; background: linear-gradient(135deg, #7f1d1d 0%, #b42318 54%, #d97706 100%); border-radius: 8px; }}
    h1 {{ margin: 0; font-size: clamp(24px, 4vw, 38px); letter-spacing: 0; }}
    section {{ margin-top: 18px; padding: 22px; background: #fff; border: 1px solid #d9dee8; border-radius: 8px; }}
    .grid {{ display: grid; grid-template-columns: repeat(4, minmax(0, 1fr)); gap: 12px; }}
    .card {{ padding: 14px; border: 1px solid #d9dee8; border-radius: 8px; background: #fbfcff; }}
    .label {{ color: #5f6877; font-size: 13px; }}
    .value {{ margin-top: 6px; font-size: 22px; font-weight: 700; overflow-wrap: anywhere; }}
    img {{ display: block; width: 100%; height: auto; border: 1px solid #d9dee8; border-radius: 8px; }}
    table {{ width: 100%; border-collapse: collapse; border: 1px solid #d9dee8; }}
    th, td {{ padding: 12px; border-bottom: 1px solid #d9dee8; text-align: left; }}
    th {{ background: #f7f9fc; }}
    .notice {{ padding: 14px; border: 1px solid #f2cf8b; border-radius: 8px; color: #8a5a00; background: #fff6dd; font-weight: 700; }}
    @media (max-width: 760px) {{ .grid {{ grid-template-columns: 1fr; }} main {{ width: calc(100% - 20px); }} }}
  </style>
</head>
<body>
  <main>
    <header><h1>贵州茅台（600519.SH）量化交易数据引擎看板</h1><p>基于过去一年日行情数据生成。</p></header>
    <section><h2>股票基本信息</h2><p>股票名称：贵州茅台；代码：600519.SH；市场：上海证券交易所 A 股主板；数据时间范围：{actual_start} 至 {actual_end}</p></section>
    <section><h2>数据概览</h2><div class="grid">
      <div class="card"><div class="label">总交易日数量</div><div class="value">{len(df)}</div></div>
      <div class="card"><div class="label">最高收盘价</div><div class="value">{max_close:.2f}</div><div>{max_date}</div></div>
      <div class="card"><div class="label">最低收盘价</div><div class="value">{min_close:.2f}</div><div>{min_date}</div></div>
      <div class="card"><div class="label">平均收盘价</div><div class="value">{avg_close:.2f}</div></div>
      <div class="card"><div class="label">起始日期</div><div class="value">{actual_start}</div></div>
      <div class="card"><div class="label">结束日期</div><div class="value">{actual_end}</div></div>
      <div class="card"><div class="label">区间首日收盘价</div><div class="value">{first_close:.2f}</div></div>
      <div class="card"><div class="label">区间末日收盘价</div><div class="value">{last_close:.2f}</div><div>{change_pct:.2f}%</div></div>
    </div></section>
    <section><h2>收盘价走势图</h2><img src="close_price_curve.png" alt="贵州茅台 600519.SH 过去一年每日收盘价走势"></section>
    <section><h2>数据质量检查表</h2><table><thead><tr><th>检查项目</th><th>结果</th><th>说明</th></tr></thead><tbody>{rows}</tbody></table></section>
    <section><h2>简短中文分析</h2><p>样本期首日收盘价为 {first_close:.2f} 元，末日收盘价为 {last_close:.2f} 元，区间变化约为 {change_pct:.2f}%。走势图可用于观察趋势和可能异常，但不能单独作为交易决策依据。</p><div class="notice">本看板仅用于课程学习和数据分析，不构成投资建议。</div></section>
  </main>
</body>
</html>'''
    HTML_PATH.write_text(html, encoding="utf-8")
    return HTML_PATH

make_dashboard(df, quality_df)

![贵州茅台收盘价曲线](close_price_curve.png)

上图展示贵州茅台过去一年每日收盘价走势，可用于观察整体趋势、阶段性波动和可能跳变。图形只是数据检查工具之一，不构成投资建议。

`600519_SH_dashboard.html` 已作为可直接在浏览器中打开的 HTML 看板生成。

## 总结与反思

本次 CSV 可以作为后续计算收益率、均线、波动率和简单回测的基础。但在真实策略研究中，还需要继续核查复权、停牌、交易成本和不同数据源差异。智能体可以帮助完成代码、检查和排版，但数据是否可用、异常如何处理、规则是否合理，仍应由人负责判断。